# Module Imports

In [ ]:
import asyncio
from asyncio import Semaphore
from bs4 import BeautifulSoup, Comment
import aiohttp
import requests

In [1]:
from utils.scraper import scrape_sitemap

from utils.utils import generate_directories


In [2]:
# Generate directories needed for project
generate_directories()

### Scrape Sitemap

In [ ]:
sitemap_url = 'https://wildfrostwiki.com/sitemap.xml'
result = scrape_sitemap(sitemap_url)

In [ ]:
result

In [ ]:
test_urls = result[2:4]

In [ ]:
urls_to_scrape = [url['url'] for url in test_urls]


In [ ]:
urls_to_scrape

### Scrape Links in Sitemap

In [ ]:
from typing import List
import re
async def scrape_single_url(session: aiohttp.ClientSession, semaphore: Semaphore, url: str):
    """
    Scrape a single URL with semaphore protection

    Args:
        session: aiohttp session
        semaphore: Limites concurrent requests
        url: URL to scrape

    Returns:
        html_output
    """

    async with semaphore:
        try:
            async with session.get(url) as response:
                response.raise_for_status()
                
                # Parse through the html
                html_output = await response.text()
                
                # TODO: Should turn this process below into a function really
                soup = BeautifulSoup(html_output, 'html.parser')
                
                # Remove the comments from the HTML
                comments = soup.find_all(string=lambda text:isinstance(text, Comment))
                for comment in comments:
                    comment.extract()

                # Extract title & remove characters that are invalid in filenames
                title = soup.find('title').text if soup.find('title') else "untitled"
                sanitized_title = re.sub(r'[\\/:*?"<>|]', '', title)
                
                if not sanitized_title:
                    sanitized_title = "output"

                filename = f"{sanitized_title}.html"
                # save the html
                with open(filename, 'w', encoding='utf-8') as f:
                    f.write(soup.prettify())
                
        
        except Exception as e:
            print(f"Error has occurred for url: {url}\nError: {e}")


async def scrape_multiple_urls(urls:List[str], max_concurrent: int = 5):
    """
    Scrape multiple urls

    Args:
        urls: List of URLs to scrape
        max_concurrent: Maximum number of simultaneous requests
    Returns:
        List of scraped content with metadata to save locally
    """
    semaphore = Semaphore(max_concurrent)

    # This creates a single aiohttp session to manage the connection pool manager
    # 1. ClientSession is created and opened
    # 2. Connection pool is initialized
    # 3. Your code runs here with 'session'
    # 4. When this block exits, session.close() is automatically called
    # 5. All connections are properly closed

    # Reuses connections when possible
    # Manages cookies across requests
    # Handles connection pooling automatically
    # Manages timeouts and retries
    # Properly closes connections when done
    async with aiohttp.ClientSession() as session:
        tasks = [scrape_single_url(session, semaphore, url) for url in urls]

        await asyncio.gather(*tasks)


# TODO Take results and then save each one

In [ ]:
urls_to_scrape

In [ ]:
await scrape_multiple_urls(urls_to_scrape)

In [ ]:
# DON'T do this - creates a new session for each request!
async def bad_fetch(url):
    async with aiohttp.ClientSession() as session:  # New session every time!
        async with session.get(url) as response:
            return await response.text()

# This is inefficient - no connection reuse
tasks = [bad_fetch(url) for url in urls_to_scrape]

In [ ]:
test_url = urls_to_scrape[0]

In [ ]:
async with aiohttp.ClientSession() as session:
    async with session.get(test_url) as response:
        response.raise_for_status()
        html_output = await response.text()
        soup = BeautifulSoup(html_output, 'html.parser')

In [ ]:
with open('Naked Gnome - Wildfrost Wiki.html', 'r', encoding='utf-8') as f:
    data = f.read()

### Given any HTML page, scrape the key information as metadata

In [ ]:
from bs4 import BeautifulSoup, Tag

def scrape_wiki_page_iterative(html_content):
    """
    Scrapes content from a BeautifulSoup element iteratively,
    collecting text from specified target tags.
    """
    soup = BeautifulSoup(html_content, 'html.parser')

    # Correctly target the div containing the parser output
    content_div = soup.find('div', {'class': 'mw-parser-output'})
    data = {}
    current_section = None
    
    if not content_div:
        # If the mw-parser-output div isn't found, try the parent div
        # as a fallback. This adds robustness to the function.
        content_div = soup.find('div', {'id': 'mw-content-text'})
        if not content_div:
            return {"error": "Main content div not found."}

    # Iterate through all direct children of the content div
    for element in content_div.children:
        # Check if the element is a BeautifulSoup Tag
        if isinstance(element, Tag):
            # If the element is a heading, start a new section
            if element.name in ['h1', 'h2', 'h3', 'h4']:
                section_title = element.get_text(strip=True)
                data[section_title] = ""
                current_section = section_title
            # If it's a target content tag and a section is defined, append its text
            elif current_section and element.name in ['p', 'ul', 'li', 'table']:
                content = element.get_text(strip=True)
                if content:
                    data[current_section] += content + "\n"
    return data

def scrape_wiki_page(html_content):
    """
    Main function to initiate the scraping process on a wiki page.
    """
    return scrape_wiki_page_iterative(html_content)

In [ ]:
scrape_wiki_page(data)